# SARBLOH Experimentation: GLM-5.3-Flash on llama.cpp, 3-hour run on the 25 public games (8.25 hours on Submit)

The agent is `Sarbloh-Experimentation/harness/`, embedded in this notebook by `scripts/kaggle_push.py glm` (no agent
dataset; the source cell prints its git SHA and whether the tree was dirty). The model is GLM-5.3-Flash (320B MoE, 18B
active, images on) in unsloth's UD-IQ4_XS GGUF (158 GB), served by llama.cpp b11878 (`harness/llm/glm.py`): the
experts that do not fit the 96 GB card stay in CPU RAM and run on the CPU, everything else on the GPU (`--fit`).
10 slots, 131072 tokens each, xhigh effort (the template's "max") on every level. A profile without images is the
fallback; no other model.

The whole notebook has a cap from the first cell, server start included: 3 hours on Save & Run, 8.25 hours on Submit.
Save & Run plays the 25 public games offline with tracing on. After the run, section 5 prints the server logs: memory
placement, start time, prompt and decode speed, and the prefix cache.

Attach: the competition (ARC wheels + games + the RTX Pro 6000), the model `kemake/glm53-flash-unsloth-public`
(Gguf / ud-iq4-xs / 1) and the dataset `banwait13/llamacpp-cuda-b11878`. Internet **off**.

## 0. CONFIG: the experiment id and its config diff (overrides of `harness/config.py`)

In [ ]:
import os, time
NOTEBOOK_START = time.time()

EXPERIMENT = "glm53_flash_llamacpp_xhigh_3h"     # the ledger label
# Submit to Competition reruns this notebook on the hidden games with KAGGLE_IS_COMPETITION_RERUN set. Its outputs are
# never seen, so tracing is off there. Save & Run keeps it on: transcripts, pictures, trace.md, reports and
# sft_levels.jsonl for LoRA SFT.
SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
TRACING = not SUBMISSION
# A cap from the first cell: 3 hours on Save & Run, 8.25 hours on Submit. The server start (a 158 GB model from the
# Kaggle input) and the 5-minute teardown come out of it. Every game starts at once; the priority scheduler lets 10
# hold a llama.cpp slot, each for 20 LLM calls before a waiting game may take it.
BUDGET_S = (8.25 if SUBMISSION else 3) * 3600.0
PLAY_S = BUDGET_S                    # no per-game cap: every game plays until the budget's teardown point
GAMES = None                         # every game: the 25 public ones offline, every hidden one on the rerun
SLOTS, QUANTUM = 10, 20   # QUANTUM: requests a game keeps its slot across turns (cache hit ~1 - 1/QUANTUM)
# A reply is slower than on Flash-Next (part of the experts runs on the CPU): one 12k-token reply at xhigh can take
# well over the 15-minute default, so a request may take an hour before it counts as failed.
REQUEST_TIMEOUT_S = 3600.0
EXPERIMENT_OVERRIDES = {
    "games": GAMES, "concurrency": SLOTS, "notebook_budget_s": BUDGET_S, "game_wall_s": PLAY_S,
    "scheduler": {"enabled": True, "slots": SLOTS, "quantum_calls": QUANTUM},
    "agent": {"limits": {"max_turns": 1000000, "max_output_tokens": 10000000000},
              # xhigh thinking on every level (GLM's template calls it "max")
              "reasoning_effort": {"early": "xhigh", "later": "xhigh"},
              "request_timeout_s": REQUEST_TIMEOUT_S,
              # cell output cut to about 2000 tokens (8000 chars at ~4 chars a token)
              "tool_output_chars": 4096,
              # the board as letters in the messages: whole at a level start and after a drain, else the changed region
              "perception": {"ascii": False}},
    "llm": {"request_timeout_s": REQUEST_TIMEOUT_S},
}

BASE = {
    "model": "glm",                   # harness/llm/glm.py: GLM-5.3-Flash on llama.cpp, experts split GPU / CPU RAM
    "model_fallback": [],             # no fallback model
    "experiment": EXPERIMENT,
    "teardown_reserve_s": 300.0,
    "record": True,                   # ARC SDK recording + agent step refs -> prime_run/trace.md (off on the rerun)
    "tracing": TRACING,               # transcripts + pictures + harness/trace.py (reports, sft_levels.jsonl)
    # None = glm.py's chain: with images, then text only
    "server": {"profile_chain": None},
}

def merge(a, b):
    out = dict(a)
    for k, v in b.items():
        out[k] = merge(out[k], v) if isinstance(v, dict) and isinstance(out.get(k), dict) else v
    return out

OVERRIDES = merge(BASE, EXPERIMENT_OVERRIDES)
print(f"submission={SUBMISSION} budget={BUDGET_S / 3600:.2f}h games={'all' if GAMES is None else len(GAMES)} "
      f"tracing={OVERRIDES['tracing']} model={OVERRIDES['model']}")
AGENT_DIR = "Sarbloh-Experimentation"   # the agent folder in the embedded source

## 1. ARC runtime (competition wheels, offline) and the agent source (embedded)

In [ ]:
import os, subprocess, sys
from pathlib import Path

COMPETITION_DIR = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3")
# arc-agi pins its own Pillow; installed over the one Kaggle already loaded, the two mix ("cannot import name '_Ink'
# from 'PIL._typing'"). So install arc-agi and arcengine without their dependencies (Kaggle has them all) and keep
# Kaggle's Pillow. Only if arc-agi then fails to import, install it with its dependencies.
_pip = [sys.executable, "-m", "pip", "install", "--quiet", "--no-index", "--no-warn-conflicts", "--disable-pip-version-check",
        "--find-links", str(COMPETITION_DIR / "arc_agi_3_wheels")]
subprocess.check_call(_pip + ["--no-deps", "arc-agi", "arcengine"], stdout=subprocess.DEVNULL)
_probe = "import arc_agi, arcengine, PIL; from PIL import Image, ImageDraw, ImageFont; print('Pillow', PIL.__version__)"
_check = subprocess.run([sys.executable, "-c", _probe], capture_output=True, text=True)
if _check.returncode != 0:
    print("arc-agi without its dependencies failed, installing them:", _check.stderr[-1500:])
    subprocess.check_call(_pip + ["arc-agi"], stdout=subprocess.DEVNULL)
    _check = subprocess.run([sys.executable, "-c", _probe], capture_output=True, text=True)
print(_check.stdout.strip() or _check.stderr[-1500:])
import importlib
importlib.invalidate_caches()
for _m in [m for m in sys.modules if m == "PIL" or m.startswith("PIL.")]:
    del sys.modules[_m]
from PIL import Image, ImageDraw, ImageFont  # a Pillow mix-up fails here, at once, not at the image smoke test

# @@AGENT_SOURCE@@
import base64, io, shutil, tarfile
AGENT_ROOT = Path("/tmp/sarbloh-src")
shutil.rmtree(AGENT_ROOT, ignore_errors=True)
AGENT_ROOT.mkdir(parents=True)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(AGENT_TGZ)), mode="r:gz") as _tf:
    _tf.extractall(AGENT_ROOT, filter="data")
for p in (AGENT_ROOT / AGENT_DIR, AGENT_ROOT):
    sys.path.insert(0, str(p))
os.environ["PRIME_GIT_SHA"] = AGENT_SOURCE_META["git_sha"] + ("-dirty" if AGENT_SOURCE_META["dirty"] else "")
print("agent source:", {k: v for k, v in AGENT_SOURCE_META.items() if k != "files"}, len(AGENT_SOURCE_META["files"]), "files")

## 2. Machine: GPU, RAM, CPU, disks, and the attached model and llama.cpp

In [ ]:
for cmd in ("nvidia-smi", "free -g", "nproc", "lscpu | grep -E 'Model name|Socket|Core|Thread|NUMA node|L3' | cut -c1-160",
            "df -h /tmp /kaggle/working", "ls -laR /kaggle/input/models 2>/dev/null | head -40",
            "find /kaggle/input -maxdepth 4 -name '*.tar.gz' 2>/dev/null"):
    print("$", cmd)
    print(subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout)

## 3. Run: server up (profile chain + smoke tests), play the games, teardown

In [ ]:
from harness.run import main

SUMMARY = main(OVERRIDES, notebook_start=NOTEBOOK_START)

## 4. Summary

In [ ]:
import json
print(json.dumps(SUMMARY, indent=1, default=str))
print(f"notebook wall: {time.time() - NOTEBOOK_START:.0f}s")

## 5. Server logs: memory placement, start, speed, prefix cache

In [ ]:
import re, subprocess
print(subprocess.run("free -g; nvidia-smi --query-gpu=memory.used,memory.total --format=csv", shell=True,
                     capture_output=True, text=True).stdout)
ev = next(Path("/kaggle/working").rglob("server_events.jsonl"), None)
if ev is not None:
    for line in ev.read_text(encoding="utf-8").splitlines():
        e = json.loads(line)
        if e["event"] != "ram":
            print(json.dumps(e, default=str)[:1500])
log_path = next(Path("/kaggle/working").rglob("llm-server.log"), None)
log = log_path.read_text(encoding="utf-8", errors="replace") if log_path else ""
print(f"\n==== {log_path}: {len(log.splitlines())} lines")
KEYS = ("fit", "offload", "CUDA0 model buffer", "CPU_Mapped model buffer", "CPU model buffer", "KV", "kv_unified",
        "compute buffer", "n_ctx", "n_parallel", "mmproj", "clip_", "chat template", "thinking", "load_tensors",
        "model loaded", "server is listening", "error", "Error", "failed", "out of memory")
for line in log.splitlines():
    if any(k in line for k in KEYS) and "slot" not in line[:40]:
        print("  " + line[:300])
# llama-server prints per request: "prompt eval time = X ms / N tokens" and "eval time = X ms / N tokens", and the
# tokens it reused from the slot ("n_past" / "cache") before the prompt.
PROMPT = re.compile(r"prompt eval time =\s*([0-9.]+) ms /\s*(\d+) tokens")
GEN = re.compile(r"\s eval time =\s*([0-9.]+) ms /\s*(\d+) tokens")
p = [(float(a), int(b)) for a, b in PROMPT.findall(log)]
g = [(float(a), int(b)) for a, b in GEN.findall(log)]
if p:
    print(f"\nprompts: {len(p)} requests, {sum(n for _, n in p)} tokens prefilled, "
          f"{sum(n for _, n in p) / max(1e-9, sum(ms for ms, _ in p) / 1000):.0f} tok/s per request while prefilling")
if g:
    print(f"replies: {len(g)} requests, {sum(n for _, n in g)} tokens, "
          f"{sum(n for _, n in g) / max(1e-9, sum(ms for ms, _ in g) / 1000):.1f} tok/s per request while decoding")
reuse = re.findall(r"n_past = (\d+).*?(?:n_tokens|task.n_tokens) = (\d+)", log)
if reuse:
    kept, total = sum(int(a) for a, _ in reuse), sum(int(b) for _, b in reuse)
    print(f"prefix reuse: {kept}/{total} prompt tokens kept from the slot ({kept / max(1, total):.1%})")
print("\n==== first 120 lines")
print("\n".join(log.splitlines()[:120]))
print("\n==== last 150 lines")
print("\n".join(log.splitlines()[-150:]))

## 6. Step trace: per game `steps.md` (turn by turn), `steps.jsonl`, SDK recording, SDK scorecard

In [ ]:
trace_md = next(Path("/kaggle/working").rglob("prime_run/trace.md"), None)
if trace_md is None:
    print("no trace.md (tracing off)" if not OVERRIDES["tracing"] else "no trace.md (run crashed before results.json?)")
else:
    print(trace_md.read_text(encoding="utf-8"))
    run_dir = trace_md.parent
    for f in sorted(run_dir.rglob("*")):
        if f.is_file() and f.suffix in (".md", ".jsonl", ".json"):
            print(f"{f.stat().st_size:>10}  {f.relative_to(run_dir)}")
    frames = [f for f in run_dir.rglob("frames/*.png")]
    print(f"{sum(f.stat().st_size for f in frames):>10}  {len(frames)} pictures in games/*/frames/")